# Dashboard de Dinâmica de Estruturas - SDOF
### Resposta ao Degrau (Sistema Amortecido)

Esta ferramenta interativa permite analisar a resposta transiente e o estado estacionário de um sistema de um grau de liberdade. 
A solução implementada é baseada no formalismo de **Chopra (2017)** para sistemas subamortecidos:

$$u(t) = \frac{F_0}{k} \left[ 1 - e^{-\zeta \omega_n t} \left( \cos(\omega_d t) + \frac{\zeta}{\sqrt{1-\zeta^2}} \sin(\omega_d t) \right) \right]$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout, HBox, VBox, Output
import ipywidgets as widgets

%matplotlib inline

def calculate_response(F0, k, m, zeta, t_max_factor=6):
    # Propriedades Dinâmicas
    omega_n = np.sqrt(k/m)
    u_st = F0/k
    T_n = 2 * np.pi / omega_n
    t = np.linspace(0, t_max_factor * T_n, 1200)
    
    if zeta < 1.0:
        wd = omega_n * np.sqrt(1 - zeta**2)
        expo = np.exp(-zeta * omega_n * t)
        u_t = u_st * (1 - expo * (np.cos(wd * t) + (zeta/np.sqrt(1-zeta**2)) * np.sin(wd * t)))
        # Envelopes
        env_up = u_st + (u_st/np.sqrt(1-zeta**2)) * expo
        env_low = u_st - (u_st/np.sqrt(1-zeta**2)) * expo
    else:
        u_t = u_st * (1 - np.exp(-omega_n * t) * (1 + omega_n * t))
        env_up, env_low = None, None
        
    return t, u_t, u_st, env_up, env_low

def update_plot(zeta, k, m, F0):
    t, u_t, u_st, env_up, env_low = calculate_response(F0, k, m, zeta)
    
    plt.figure(figsize=(11, 5), dpi=100)
    plt.style.use('bmh')
    
    plt.plot(t, u_t, color='#2563eb', lw=2, label='Resposta Dinâmica $u(t)$')
    plt.axhline(u_st, color='#ef4444', linestyle='--', alpha=0.8, label=f'Desloc. Estático $u_{{st}}$ = {u_st:.4f}m')
    
    if env_up is not None and zeta > 0:
        plt.plot(t, env_up, color='gray', linestyle=':', alpha=0.5, label='Envelope de Amortecimento')
        plt.plot(t, env_low, color='gray', linestyle=':', alpha=0.5)
    
    plt.title(f'Resposta SDOF (F0={F0}N, k={k}N/m, m={m}kg)', fontsize=12)
    plt.xlabel('Tempo (s)')
    plt.ylabel('Deslocamento $u$ (m)')
    plt.legend(loc='upper right', frameon=True, facecolor='white')
    plt.ylim(0, u_st * 2.1)
    plt.tight_layout()
    plt.show()

# Controles Interativos com Layout para Voila
zeta_slider = FloatSlider(min=0, max=0.99, step=0.01, value=0.05, description='Amortecimento $\zeta$')
k_slider = FloatSlider(min=1000, max=20000, step=500, value=5000, description='Rigidez $k$ (N/m)')
m_slider = FloatSlider(min=1, max=100, step=1, value=20, description='Massa $m$ (kg)')
f0_slider = FloatSlider(min=100, max=5000, step=100, value=1000, description='Força $F_0$ (N)')

out = widgets.interactive_output(update_plot, {
    'zeta': zeta_slider, 
    'k': k_slider, 
    'm': m_slider, 
    'F0': f0_slider
})

ui = VBox([
    HBox([zeta_slider, k_slider]),
    HBox([m_slider, f0_slider]),
    out
])

display(ui)